<a href="https://colab.research.google.com/github/bader1919/pf-pipeline/blob/master/notebooks/pf_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PF Bahrain — explore the data with DuckDB

Reads the daily Parquet files from Cloudflare R2 (bucket `pf-pipeline`) — every listing, every day since 9 June 2026.

**How to use:** run the cells top to bottom (Shift+Enter).
- Step 1 asks for your R2 keys (they are never saved in this file).
- Step 2 downloads a local copy the first time (~1 GB, a few minutes). After that every query takes about a second, and re-running Step 2 only fetches the new days.
- Then try the example questions, or write your own SQL at the end.

In [ ]:
%pip install -q duckdb pandas matplotlib

In [ ]:
from google.colab import userdata
userdata.get('R2_ACCOUNT_ID', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY')

## Step 1 — connect to R2
Uses the environment variables `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` if set; otherwise asks for them.

In [ ]:
import os, getpass
import duckdb

BASE = os.environ.get("PF_PARQUET_BASE", "r2://pf-pipeline/parquet")
CATEGORIES = ["residential_rent", "residential_sale", "commercial_rent", "commercial_sale", "new_projects"]

con = duckdb.connect("pf_local.duckdb")   # local file next to this notebook

def key(name):
    return os.environ.get(name) or getpass.getpass(f"{name}: ")

if BASE.startswith("r2://"):
    con.execute(f"""CREATE OR REPLACE SECRET r2 (TYPE r2,
        ACCOUNT_ID '{key("R2_ACCOUNT_ID")}',
        KEY_ID '{key("R2_ACCESS_KEY_ID")}',
        SECRET '{key("R2_SECRET_ACCESS_KEY")}')""")
print("connected:", BASE)

## Step 2 — local copy (first run downloads everything; later runs only new days)

In [ ]:
for cat in CATEGORIES:
    src = f"read_parquet('{BASE}/{cat}/*.parquet', filename=true, union_by_name=true)"
    exists = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = ?", [cat]).fetchone()[0]
    if not exists:
        con.execute(f"CREATE TABLE {cat} AS SELECT * EXCLUDE (filename) FROM {src}")
    else:
        last = con.execute(f"SELECT max(snapshot_date) FROM {cat}").fetchone()[0]
        con.execute(f"INSERT INTO {cat} BY NAME SELECT * EXCLUDE (filename) FROM {src} "
                    f"WHERE filename > '{BASE}/{cat}/{last}.parquet'")
    days, rows = con.execute(f"SELECT count(DISTINCT snapshot_date), count(*) FROM {cat}").fetchone()
    print(f"{cat:18} {days:4} days {rows:>10,} rows")

# one view over the four listing categories (new_projects has its own columns)
con.execute("""CREATE OR REPLACE VIEW listings AS
    SELECT 'residential_rent' AS category, * FROM residential_rent
    UNION ALL BY NAME SELECT 'residential_sale' AS category, * FROM residential_sale
    UNION ALL BY NAME SELECT 'commercial_rent' AS category, * FROM commercial_rent
    UNION ALL BY NAME SELECT 'commercial_sale' AS category, * FROM commercial_sale""")

def q(sql):
    """Run SQL, return a pandas table."""
    return con.sql(sql).df()

## Good to know
- Every column is stored as **text**, exactly as scraped. Turn numbers into numbers with `TRY_CAST(price_value AS DOUBLE)` (junk becomes NULL instead of an error).
- One row = one listing on one day (`snapshot_date`). A listing seen 100 days has 100 rows.
- Rent prices are **monthly**, in BHD. `bedrooms_value` is `studio`, `1`, `2`, … (commercial listings have none).
- Tables: `residential_rent`, `residential_sale`, `commercial_rent`, `commercial_sale`, `new_projects`, and the view `listings` (all four listing categories).
- Days with a partial scrape (14 and 15 July) are not included.

## Q1 — listings per day and category

In [ ]:
q("""SELECT snapshot_date, category, count(*) AS listings
     FROM listings GROUP BY ALL ORDER BY snapshot_date DESC, category LIMIT 12""")

## Q2 — median monthly rent by area and bedrooms (latest day, apartments)

In [ ]:
q("""SELECT area_name, bedrooms_value AS beds, count(*) AS listings,
            median(TRY_CAST(price_value AS DOUBLE))::INT AS median_rent_bhd
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
       AND property_type = 'Apartment' AND bedrooms_value IN ('studio', '1', '2', '3')
     GROUP BY ALL HAVING count(*) >= 20
     ORDER BY area_name, beds""")

## Q3 — trend: median 2-bed apartment rent per day in one area
Change `AREA` to any `area_name` from Q2.

In [ ]:
AREA = "Al Juffair"
trend = q(f"""SELECT snapshot_date, count(*) AS listings,
                median(TRY_CAST(price_value AS DOUBLE)) AS median_rent_bhd
         FROM residential_rent
         WHERE area_name = '{AREA}' AND property_type = 'Apartment' AND bedrooms_value = '2'
         GROUP BY 1 ORDER BY 1""")
trend.plot(x="snapshot_date", y="median_rent_bhd", title=f"Median 2-bed apartment rent — {AREA}", legend=False)
trend.tail()

## Q4 — new and removed listings per day
New = first day a listing appears. Removed = its last day (excluding the newest day, where nothing has had a chance to disappear yet).

In [ ]:
q("""WITH life AS (
         SELECT pf_id, category, min(snapshot_date) AS first_seen, max(snapshot_date) AS last_seen
         FROM listings GROUP BY ALL),
       days AS (SELECT DISTINCT snapshot_date AS d FROM listings),
       bounds AS (SELECT min(d) AS first_day, max(d) AS last_day FROM days)
     SELECT d AS day,
            count(*) FILTER (WHERE first_seen = d AND d > first_day) AS new_listings,
            count(*) FILTER (WHERE last_seen  = d AND d < last_day)  AS removed_listings
     FROM days, bounds, life
     WHERE first_seen = d OR last_seen = d
     GROUP BY d ORDER BY d DESC LIMIT 14""")

## Q5 — areas with the most new rental listings in the last 30 days

In [ ]:
q("""WITH first AS (SELECT pf_id, any_value(area_name) AS area_name, min(snapshot_date) AS first_seen
                    FROM residential_rent GROUP BY pf_id)
     SELECT area_name, count(*) AS new_listings
     FROM first
     WHERE first_seen > (SELECT max(snapshot_date) - 30 FROM residential_rent)
     GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")

## Q6 — price changes: rental listings whose price moved (latest day vs. about 30 days earlier)

In [ ]:
q("""WITH now AS (SELECT pf_id, area_name, title, TRY_CAST(price_value AS DOUBLE) AS price_now
                  FROM residential_rent WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)),
          before AS (SELECT pf_id, TRY_CAST(price_value AS DOUBLE) AS price_before
                     FROM residential_rent
                     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent  -- closest day 30+ days back
                                            WHERE snapshot_date <= (SELECT max(snapshot_date) - 30 FROM residential_rent)))
     SELECT area_name, count(*) AS changed,
            count(*) FILTER (WHERE price_now < price_before) AS cut,
            count(*) FILTER (WHERE price_now > price_before) AS raised
     FROM now JOIN before USING (pf_id)
     WHERE price_now <> price_before
     GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")

## Q7 — sale prices: median price per sqm by area (latest day, apartments)

In [ ]:
q("""SELECT area_name, count(*) AS listings,
            median(TRY_CAST(price_value AS DOUBLE))::INT AS median_price_bhd,
            median(TRY_CAST(price_value AS DOUBLE) / NULLIF(TRY_CAST(size_value AS DOUBLE), 0))::INT AS median_bhd_per_size_unit,
            any_value(size_unit) AS size_unit
     FROM residential_sale
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_sale) AND property_type = 'Apartment'
     GROUP BY 1 HAVING count(*) >= 20 ORDER BY median_bhd_per_size_unit DESC""")

# Simple examples — learn the basics
Each one is a small, readable query. Copy one, change a word, run it.

### E1 — look at 5 rows (see what the data looks like)

In [ ]:
q("""SELECT snapshot_date, area_name, property_type, bedrooms_value, price_value, title
     FROM residential_rent
     LIMIT 5""")

### E2 — filter: 2-bed apartments for rent in Seef, latest day, cheapest first

In [ ]:
q("""SELECT title, TRY_CAST(price_value AS DOUBLE) AS rent_bhd, size_value, furnished
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
       AND area_name = 'Seef' AND property_type = 'Apartment' AND bedrooms_value = '2'
     ORDER BY rent_bhd
     LIMIT 10""")

### E3 — count: how many listings of each property type are for sale today?

In [ ]:
q("""SELECT property_type, count(*) AS listings
     FROM residential_sale
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_sale)
     GROUP BY property_type
     ORDER BY listings DESC""")

### E4 — min / average / max: villa rents by area (latest day)
Watch the max column: a few agents put a **sale price in the rent section** (e.g. 160,000 BHD/month). Those outliers pull the *average* up — that is why the other examples use the *median*.

In [ ]:
q("""SELECT area_name, count(*) AS villas,
            min(TRY_CAST(price_value AS DOUBLE)) AS cheapest,
            round(avg(TRY_CAST(price_value AS DOUBLE))) AS average,
            max(TRY_CAST(price_value AS DOUBLE)) AS most_expensive
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent) AND property_type = 'Villa'
     GROUP BY area_name HAVING count(*) >= 10
     ORDER BY average DESC""")

### E5 — text search: listings that mention 'sea view' in the title

In [ ]:
q("""SELECT area_name, bedrooms_value, TRY_CAST(price_value AS DOUBLE) AS rent_bhd, title
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
       AND title ILIKE '%sea view%'
     ORDER BY rent_bhd
     LIMIT 10""")

### E6 — furnished or not: share of furnished 1-bed apartments by area

In [ ]:
q("""SELECT area_name, count(*) AS listings,
            round(100.0 * count(*) FILTER (WHERE furnished = 'YES') / count(*)) AS pct_furnished
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
       AND property_type = 'Apartment' AND bedrooms_value = '1'
     GROUP BY area_name HAVING count(*) >= 30
     ORDER BY pct_furnished DESC""")

### E7 — compare two areas side by side (median rent by bedrooms)

In [ ]:
A, B = "Al Juffair", "Seef"
q(f"""SELECT bedrooms_value AS beds,
             median(TRY_CAST(price_value AS DOUBLE)) FILTER (WHERE area_name = '{A}') AS "{A}",
             median(TRY_CAST(price_value AS DOUBLE)) FILTER (WHERE area_name = '{B}') AS "{B}"
      FROM residential_rent
      WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent) AND property_type = 'Apartment'
        AND bedrooms_value IN ('studio', '1', '2', '3')
      GROUP BY beds ORDER BY beds""")

### E8 — biggest brokers: who has the most rental listings today?

In [ ]:
q("""SELECT broker_name, count(*) AS listings
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
     GROUP BY broker_name
     ORDER BY listings DESC
     LIMIT 10""")

### E9 — one listing's history: every day it was online and its price
Picks a long-running listing; or set `PF_ID = '...'` to any `pf_id` from the queries above.

In [ ]:
PF_ID = q("SELECT pf_id FROM residential_rent GROUP BY pf_id ORDER BY count(*) DESC LIMIT 1").pf_id[0]  # a long-running listing
q(f"""SELECT snapshot_date, TRY_CAST(price_value AS DOUBLE) AS price, title
      FROM listings
      WHERE pf_id = '{PF_ID}'
      ORDER BY snapshot_date""")

### E10 — sale vs. rent: how many years of rent buys an apartment? (rough, by area)

In [ ]:
q("""WITH latest AS (SELECT max(snapshot_date) AS d FROM residential_rent),
          rent AS (SELECT area_name, median(TRY_CAST(price_value AS DOUBLE)) AS monthly_rent
                   FROM residential_rent, latest WHERE snapshot_date = d
                     AND property_type = 'Apartment' AND bedrooms_value = '2' GROUP BY 1),
          sale AS (SELECT area_name, median(TRY_CAST(price_value AS DOUBLE)) AS price
                   FROM residential_sale, latest WHERE snapshot_date = d
                     AND property_type = 'Apartment' AND bedrooms_value = '2' GROUP BY 1)
     SELECT area_name, monthly_rent, price,
            round(price / (monthly_rent * 12), 1) AS years_of_rent,
            round(100 * monthly_rent * 12 / price, 1) AS gross_yield_pct
     FROM rent JOIN sale USING (area_name)
     ORDER BY gross_yield_pct DESC""")

## Your own question
All columns: run `q("DESCRIBE residential_rent")`.

In [ ]:
q("""SELECT property_type, count(*) AS listings
     FROM residential_rent
     WHERE snapshot_date = (SELECT max(snapshot_date) FROM residential_rent)
     GROUP BY 1 ORDER BY 2 DESC""")